In [15]:
import os
from datetime import datetime
import time
import requests
from pathlib import Path
import pandas as pd
import requests
from bs4 import BeautifulSoup
from lxml import etree

The following functions can be used to scrape SEC Form N-PORT data from EDGAR. There are test cases throughout the code as well as examples of how to use the full data pipeline at the end. If you wish to obtain all SEC Form N-PORT filings this script will work, but will take multiple hours to run.

In [16]:
# ------------------------------------------------------------------------------
# Configuration & Helper Functions
# ------------------------------------------------------------------------------
DEFAULT_HEADERS = {
    "User-Agent": "YourNameOrOrganization user@domain.com"  # SEC requires a descriptive User-Agent
}

def enforce_rate_limit(request_count, max_requests=10, start_time=None):
    """
    Enforces SEC EDGAR rate limit (max 10 requests per second).
    """
    if start_time is None:
        start_time = time.time()
        
    request_count += 1
    if request_count >= max_requests:
        elapsed_time = time.time() - start_time
        if elapsed_time < 1.0:
            time.sleep(1.0 - elapsed_time)
        start_time = time.time()
        request_count = 0
    return request_count, start_time

In [17]:
# ------------------------------------------------------------------------------
# Retrieving Mutual Fund Identifiers
# ------------------------------------------------------------------------------

# SEC EDGAR requires a specific User-Agent format: Sample Company Name AdminContact@domain.com
HEADERS = {
    "User-Agent": "FinancialDataPipeline contact@example.com"
}

def get_mutual_fund_universe(headers=HEADERS):
    """
    Fetches the Mutual Fund Universe JSON from SEC and returns a DataFrame
    mapping zero-padded CIK, seriesId, classId, and ticker symbol.
    """
    url = "https://www.sec.gov/files/company_tickers_mf.json"
    
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    
    json_data = response.json()
    df = pd.DataFrame(json_data['data'], columns=json_data['fields'])
    
    # Crucial: Format CIK as a 10-digit zero-padded string for SEC EDGAR compatibility
    df['cik'] = df['cik'].astype(str).str.zfill(10)
    
    # Group by seriesId to get a clean mapping per fund series
    mf_universe = df.groupby('seriesId', as_index=False).agg({
        'cik': 'first',
        'classId': 'first',
        'symbol': 'first'
    })[['cik', 'seriesId', 'classId', 'symbol']]
    
    return mf_universe

if __name__ == "__main__":
    universe_df = get_mutual_fund_universe()
    print(f"Loaded {len(universe_df)} unique fund series.")
    print(universe_df.head())

Loaded 12180 unique fund series.
          cik    seriesId     classId symbol
0  0000318478  S000000008  C000000012  DGAGX
1  0000720064  S000000016  C000000023  DRCAX
2  0000030146  S000000068  C000000105  DREVX
3  0000030162  S000000071  C000000108  DREQX
4  0000718935  S000000073  C000000110  DITEX


In [18]:
# ------------------------------------------------------------------------------
# Parses primary form N-PORT Documentation
# ------------------------------------------------------------------------------
def parse_nport_xml(xml_source, headers=HEADERS):
    """
    Parses N-PORT primary_doc.xml content or URL into a Pandas DataFrame.
    Converts all XML tags to lowercase to match the target schema exactly.
    """
    if isinstance(xml_source, str) and xml_source.startswith("http"):
        resp = requests.get(xml_source, headers=headers)
        if resp.status_code != 200:
            print(f"Failed to fetch XML from {xml_source}: HTTP {resp.status_code}")
            return pd.DataFrame()
        xml_bytes = resp.content
    elif isinstance(xml_source, bytes):
        xml_bytes = xml_source
    else:
        xml_bytes = xml_source.encode('utf-8')

    try:
        root = etree.fromstring(xml_bytes)
    except etree.XMLSyntaxError as e:
        print(f"XML Syntax Error: {e}")
        return pd.DataFrame()

    # Namespace-agnostic XPath search for holding records (<invstOrSec>)
    invst_nodes = root.xpath('.//*[local-name()="invstOrSec"]')
    
    records = []
    for node in invst_nodes:
        row = {}
        for child in node:
            # Force lowercase tag names to match target CSV schema
            tag_name = etree.QName(child).localname.lower()
            
            # 1. Handle nested <identifiers> (cusip, isin, ticker, lei, other)
            if tag_name == 'identifiers':
                for sub in child:
                    sub_tag = etree.QName(sub).localname.lower()
                    row[sub_tag] = sub.attrib.get('value') or (sub.text.strip() if sub.text else None)
            
            # 2. Handle complex nested tags (debtsec, derivativeinfo, currencyconditional, issuerconditional)
            elif len(child) > 0:
                sub_dict = {}
                for sub in child:
                    sub_tag = etree.QName(sub).localname.lower()
                    sub_val = sub.attrib.get('value') or (sub.text.strip() if sub.text else None)
                    if sub_val:
                        sub_dict[sub_tag] = sub_val
                row[tag_name] = str(sub_dict) if sub_dict else None
                
            # 3. Handle flat text elements (name, balance, units, valusd, pctval, etc.)
            else:
                row[tag_name] = child.text.strip() if child.text else None
                
        records.append(row)
        
    return pd.DataFrame(records)

if __name__ == "__main__":
    # Quick sanity check test with a sample SEC N-PORT XML URL
    sample_url = "https://www.sec.gov/Archives/edgar/data/1618627/000114554923004968/primary_doc.xml"
    df_sample = parse_nport_xml(sample_url)
    print(f"Parsed {len(df_sample)} holdings from sample XML.")
    print(df_sample.head())

Parsed 127 holdings from sample XML.
                      name                   lei  \
0          ARROW BIDCO LLC  549300YHZN08M0H3O128   
1    CD&R SMOKEY BUYER INC                   N/A   
2            CARNIVAL CORP  F1OF2ZSX47CR0BCWA982   
3  CUMULUS MEDIA NEW HLDGS  5493001D9VBAVY1OKJ46   
4            DISH DBS CORP                   N/A   

                            title      cusip          isin  \
0                 Arrow Bidco LLC  042728AA3  US042728AA35   
1           CD&R Smokey Buyer Inc  12510CAA9  US12510CAA99   
2                   Carnival Corp  143658BG6  US143658BG61   
3  Cumulus Media New Holdings Inc  23110AAA4  US23110AAA43   
4                   DISH DBS Corp  25470XAW5  US25470XAW56   

               balance units curcd               valusd        pctval  \
0  115000.000000000000    PA   USD  114754.170000000000  0.3967552449   
1  165000.000000000000    PA   USD  146231.250000000000  0.5055852472   
2  385000.000000000000    PA   USD  389658.500000000000  1

In [19]:
# ------------------------------------------------------------------------------
# Retrieves Form N-PORT links based on date range, CIK, and Series_ID
# ------------------------------------------------------------------------------
def parse_date(date_str):
    """Parses various date string formats into a standard datetime object."""
    if not date_str:
        return None
    for fmt in ("%Y-%m-%d", "%m-%d-%Y", "%Y%m%d", "%m/%d/%Y"):
        try:
            return datetime.strptime(date_str.strip(), fmt)
        except ValueError:
            pass
    return None

def get_nport_filing_urls(cik, series_id=None, start_date=None, end_date=None, fetch_all=True, headers=HEADERS):
    """
    Retrieves N-PORT filings for a given CIK or Series ID.
    
    Parameters:
      - cik: 10-digit CIK string.
      - series_id: Fund Series ID (e.g., 'S000000001').
      - start_date: Optional 'YYYY-MM-DD' filter for minimum period_of_report.
      - end_date: Optional 'YYYY-MM-DD' filter for maximum period_of_report.
      - fetch_all: If True, paginates through all historical filings on SEC EDGAR.
    """
    cik_str = str(cik).zfill(10)
    search_target = series_id if series_id else cik_str
    
    start_dt = parse_date(start_date) if start_date else None
    end_dt = parse_date(end_date) if end_date else None

    filings = []
    start_offset = 0
    page_size = 100  # EDGAR limit per HTML results page

    while True:
        url = f"https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK={search_target}&type=NPORT-P&dateb=&count={page_size}&start={start_offset}&owner=exclude"
        
        resp = requests.get(url, headers=headers)
        if resp.status_code != 200:
            print(f"EDGAR search failed for {search_target}: HTTP {resp.status_code}")
            break

        soup = BeautifulSoup(resp.content, 'html.parser')
        
        company_info = soup.find('span', {'class': 'companyName'})
        registrant_name = company_info.text.split(' CIK#')[0].strip() if company_info else ""

        # Extract filing table rows
        rows = soup.find_all('tr')
        nport_rows = [r for r in rows if 'NPORT-P' in r.get_text()]

        if not nport_rows:
            break  # End of available filings

        for row in nport_rows:
            row_text = row.get_text()
            submission_type = 'NPORT-P/A' if 'NPORT-P/A' in row_text else 'NPORT-P'
            
            doc_link = row.find('a', href=True, string=lambda s: s and 'Documents' in s)
            if not doc_link:
                continue
                
            doc_folder_url = "https://www.sec.gov" + doc_link['href']
            
            time.sleep(0.12)  # Maintain SEC rate limit (< 10 requests/sec)
            
            folder_resp = requests.get(doc_folder_url, headers=headers)
            if folder_resp.status_code != 200:
                continue
                
            folder_soup = BeautifulSoup(folder_resp.content, 'html.parser')
            
            period_div = folder_soup.find('div', string='Period of Report')
            filing_div = folder_soup.find('div', string='Filing Date')
            accepted_div = folder_soup.find('div', string='Accepted')

            period_of_report = period_div.find_next_sibling('div').get_text(strip=True) if period_div and period_div.find_next_sibling('div') else ""
            filing_date = filing_div.find_next_sibling('div').get_text(strip=True) if filing_div and filing_div.find_next_sibling('div') else ""
            accepted_date = accepted_div.find_next_sibling('div').get_text(strip=True) if accepted_div and accepted_div.find_next_sibling('div') else ""

            # Filter by period_of_report date range
            if period_of_report:
                report_dt = parse_date(period_of_report)
                if report_dt:
                    if start_dt and report_dt < start_dt:
                        continue
                    if end_dt and report_dt > end_dt:
                        continue

            xml_link = folder_soup.find('a', href=True, string=lambda s: s and s.endswith('.xml') and 'primary' in s.lower())
            
            if xml_link:
                xml_url = "https://www.sec.gov" + xml_link['href']
                filings.append({
                    'cik': cik_str,
                    'series_id': series_id,
                    'registrant_name': registrant_name,
                    'submission_type': submission_type,
                    'period_of_report': period_of_report,
                    'filing_date': filing_date,
                    'accepted_date': accepted_date,
                    'xml_url': xml_url
                })

        # Break loop if not fetching full history or if fewer than 100 rows were returned
        if not fetch_all or len(nport_rows) < page_size:
            break
            
        start_offset += page_size

    return filings

if __name__ == "__main__":
    # Test finder on CIK 0001618627
    test_filings = get_nport_filing_urls(cik="0001618627")
    print("Found Filings:", test_filings)

Found Filings: [{'cik': '0001618627', 'series_id': None, 'registrant_name': 'RBB Fund Trust', 'submission_type': 'NPORT-P', 'period_of_report': '2026-05-31', 'filing_date': '2026-07-28', 'accepted_date': '2026-07-28 11:46:39', 'xml_url': 'https://www.sec.gov/Archives/edgar/data/1618627/000119312526319545/primary_doc.xml'}, {'cik': '0001618627', 'series_id': None, 'registrant_name': 'RBB Fund Trust', 'submission_type': 'NPORT-P', 'period_of_report': '2026-05-31', 'filing_date': '2026-07-28', 'accepted_date': '2026-07-28 11:44:32', 'xml_url': 'https://www.sec.gov/Archives/edgar/data/1618627/000119312526319538/primary_doc.xml'}, {'cik': '0001618627', 'series_id': None, 'registrant_name': 'RBB Fund Trust', 'submission_type': 'NPORT-P', 'period_of_report': '2026-05-31', 'filing_date': '2026-07-28', 'accepted_date': '2026-07-28 11:44:31', 'xml_url': 'https://www.sec.gov/Archives/edgar/data/1618627/000119312526319537/primary_doc.xml'}, {'cik': '0001618627', 'series_id': None, 'registrant_name

In [20]:
# ------------------------------------------------------------------------------
# Executes functions as a data pipeline and saves CSV files to nport_output folder
# ------------------------------------------------------------------------------

TARGET_COLUMNS = [
    'name', 'lei', 'title', 'cusip', 'isin', 'balance', 'units', 'curcd', 
    'valusd', 'pctval', 'payoffprofile', 'assetcat', 'issuercat', 'invcountry', 
    'isrestrictedsec', 'fairvallevel', 'securitylending', 'other', 'issuerconditional', 
    'derivativeinfo', 'period_of_report', 'filing_date', 'accepted_date', 
    'submission_type', 'cik', 'series_id', 'registrant_name', 'ticker', 
    'currencyconditional', 'debtsec'
]

def run_nport_pipeline(
    series_ids=None,         # List of specific Series IDs, e.g., ['S000001234', 'S000056789']
    row_indices=None,        # List of specific row numbers in the universe, e.g., [0, 5, 12]
    series_limit=None,       # Maximum number of series (used as fallback if no list provided)
    start_date=None, 
    end_date=None, 
    fetch_all=True, 
    output_dir="./nport_output"
):
    """
    Runs the pipeline across mutual fund series with options to filter by specific identifiers,
    row indices, date ranges, or fetch all history.
    """
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    print("Fetching Mutual Fund Universe...")
    universe_df = get_mutual_fund_universe()
    if universe_df.empty:
        print("Failed to load universe.")
        return

    # --- FILTERING LOGIC ---
    if series_ids is not None:
        sample_universe = universe_df[universe_df['seriesId'].isin(series_ids)]
    elif row_indices is not None:
        sample_universe = universe_df.iloc[row_indices]
    elif series_limit is not None:
        sample_universe = universe_df.head(series_limit)
    else:
        sample_universe = universe_df  # Process entire universe if no limit or filter is set

    if sample_universe.empty:
        print("No matching fund series found for the specified criteria.")
        return

    # Reset index for clean loop logging
    sample_universe = sample_universe.reset_index(drop=True)

    for idx, row in sample_universe.iterrows():
        cik = row['cik']
        series_id = row['seriesId']
        fund_ticker = row.get('symbol', '')

        print(f"\n[{idx+1}/{len(sample_universe)}] Searching filings for Series: {series_id} (CIK: {cik})...")
        
        filing_records = get_nport_filing_urls(
            cik=cik, 
            series_id=series_id, 
            start_date=start_date, 
            end_date=end_date, 
            fetch_all=fetch_all
        )
        
        print(f"Found {len(filing_records)} matching filings.")

        all_series_holdings = []

        for f_idx, metadata in enumerate(filing_records):
            xml_url = metadata['xml_url']
            print(f" Parsing [{f_idx+1}/{len(filing_records)}] Period: {metadata['period_of_report']} | URL: {xml_url}")
            
            holdings_df = parse_nport_xml(xml_url)
            
            if not holdings_df.empty:
                holdings_df['period_of_report'] = metadata['period_of_report']
                holdings_df['filing_date'] = metadata['filing_date']
                holdings_df['accepted_date'] = metadata['accepted_date']
                holdings_df['submission_type'] = metadata['submission_type']
                holdings_df['registrant_name'] = metadata['registrant_name']
                holdings_df['cik'] = metadata['cik']
                holdings_df['series_id'] = metadata['series_id']
                
                if 'ticker' not in holdings_df.columns or holdings_df['ticker'].isna().all():
                    holdings_df['ticker'] = fund_ticker

                final_df = holdings_df.reindex(columns=TARGET_COLUMNS)
                all_series_holdings.append(final_df)
                
            time.sleep(0.12)

        if all_series_holdings:
            combined_df = pd.concat(all_series_holdings, ignore_index=True)
            output_file = output_path / f"nport_{series_id}.csv"
            combined_df.to_csv(output_file, index=False)
            print(f"Successfully saved {len(combined_df)} rows across {len(filing_records)} filings to {output_file}")

The following chunks of code are used as examples for how to execute the NPORT pipeline.

In [ ]:
# Pulls all NPORT filings for a specific series_id
run_nport_pipeline(series_ids=['S000000698'])

Fetching Mutual Fund Universe...

[1/1] Searching filings for Series: S000000698 (CIK: 0000811161)...
Found 28 matching filings.
 Parsing [1/28] Period: 2026-06-30 | URL: https://www.sec.gov/Archives/edgar/data/811161/000206657826002501/primary_doc.xml
 Parsing [2/28] Period: 2026-03-31 | URL: https://www.sec.gov/Archives/edgar/data/811161/000206657826001343/primary_doc.xml
 Parsing [3/28] Period: 2025-12-31 | URL: https://www.sec.gov/Archives/edgar/data/811161/000206657826000148/primary_doc.xml
 Parsing [4/28] Period: 2025-09-30 | URL: https://www.sec.gov/Archives/edgar/data/811161/000206657825001380/primary_doc.xml
 Parsing [5/28] Period: 2025-06-30 | URL: https://www.sec.gov/Archives/edgar/data/811161/000114554925053156/primary_doc.xml
 Parsing [6/28] Period: 2025-03-31 | URL: https://www.sec.gov/Archives/edgar/data/811161/000114554925032988/primary_doc.xml
 Parsing [7/28] Period: 2024-12-31 | URL: https://www.sec.gov/Archives/edgar/data/811161/000114554925007945/primary_doc.xml
 Pa

In [ ]:
# Pulls all NPORT filings for the 1st, 6th, and 11th entries in the mutual funds universe
run_nport_pipeline(row_indices=[0, 5, 10])

Fetching Mutual Fund Universe...

[1/3] Searching filings for Series: S000000008 (CIK: 0000318478)...
Found 32 matching filings.
 Parsing [1/32] Period: 2026-06-30 | URL: https://www.sec.gov/Archives/edgar/data/318478/000094040026033357/primary_doc.xml
 Parsing [2/32] Period: 2026-03-31 | URL: https://www.sec.gov/Archives/edgar/data/318478/000094040026021262/primary_doc.xml
 Parsing [3/32] Period: 2025-12-31 | URL: https://www.sec.gov/Archives/edgar/data/318478/000094040026007303/primary_doc.xml
 Parsing [4/32] Period: 2025-09-30 | URL: https://www.sec.gov/Archives/edgar/data/318478/000094040025011098/primary_doc.xml
 Parsing [5/32] Period: 2025-06-30 | URL: https://www.sec.gov/Archives/edgar/data/318478/000175272425205919/primary_doc.xml
 Parsing [6/32] Period: 2025-03-31 | URL: https://www.sec.gov/Archives/edgar/data/318478/000175272425124167/primary_doc.xml
 Parsing [7/32] Period: 2024-12-31 | URL: https://www.sec.gov/Archives/edgar/data/318478/000175272425038842/primary_doc.xml
 Pa

In [ ]:
# Pulls NPORT filings from January 1, 2022 and to December 31, 2023 for the 1st and 2nd entries in the mutual fund universe 
run_nport_pipeline(series_limit=2, start_date="2022-01-01", end_date="2023-12-31")

Fetching Mutual Fund Universe...

[1/2] Searching filings for Series: S000000008 (CIK: 0000318478)...
Found 10 matching filings.
 Parsing [1/10] Period: 2023-12-31 | URL: https://www.sec.gov/Archives/edgar/data/318478/000177569724000355/primary_doc.xml
 Parsing [2/10] Period: 2023-12-31 | URL: https://www.sec.gov/Archives/edgar/data/318478/000177569724000109/primary_doc.xml
 Parsing [3/10] Period: 2023-10-31 | URL: https://www.sec.gov/Archives/edgar/data/318478/000177569723001153/primary_doc.xml
 Parsing [4/10] Period: 2023-09-30 | URL: https://www.sec.gov/Archives/edgar/data/318478/000177569723000989/primary_doc.xml
 Parsing [5/10] Period: 2023-06-30 | URL: https://www.sec.gov/Archives/edgar/data/318478/000177569723000704/primary_doc.xml
 Parsing [6/10] Period: 2023-03-31 | URL: https://www.sec.gov/Archives/edgar/data/318478/000177569723000425/primary_doc.xml
 Parsing [7/10] Period: 2022-12-31 | URL: https://www.sec.gov/Archives/edgar/data/318478/000177569723000120/primary_doc.xml
 Pa